# Random Forest improvement experiments — Alma
The original notebook and its outputs remain unchanged. This notebook records a controlled follow-up.
**Selection protocol:** use the same 2,000-row validation IDs; choose representation and RF settings using 3-fold CV inside the 8,000 training rows; then inspect validation once for this candidate set.
Validation has already informed the research direction, so results are exploratory, not an unbiased final test.
No Kaggle submissions or test-label access are performed.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
import hashlib, json, re, time, sys
import numpy as np
import pandas as pd
import scipy
import sklearn
from scipy import sparse
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.pipeline import FeatureUnion
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import roc_auc_score

# Works from this project directory in PyCharm or from run_notebook.py.
PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "data").exists():
    raise RuntimeError("Set notebook working directory to Kaggle Project.")
DATA_DIR = PROJECT_DIR / "data"
BASELINE_DIR = PROJECT_DIR / "outputs" / "rf_20261107T193931_779056Z"
RUN_DIR = PROJECT_DIR / "outputs" / datetime.now(timezone.utc).strftime("rf_improvement_%Y%m%dT%H%M%S_%fZ")
RUN_DIR.mkdir(parents=True, exist_ok=False)
SEED = 42
RF_BASE = dict(n_estimators=200, max_depth=None, min_samples_leaf=2,
               max_features="sqrt", random_state=SEED, n_jobs=-1)
print("Output directory:", RUN_DIR)

## 1. Preserve the original split
Regenerate the original row ordering using the original seed/stratification, then verify actual IDs and dataset hash against the saved baseline.
We do not search for a more favorable validation split.

In [ ]:
df = pd.read_csv(DATA_DIR / "train.csv", dtype={"id": str})
texts = df["text"].fillna("").astype(str)
y = df["label"].to_numpy()
assert set(y) == {0, 1}
dataset_hash = hashlib.sha256((DATA_DIR / "train.csv").read_bytes()).hexdigest()
old_meta = json.loads((BASELINE_DIR / "run_metadata.json").read_text())
assert dataset_hash == old_meta["dataset_sha256"]
train_idx, val_idx = train_test_split(np.arange(len(df)), test_size=2000,
                                     random_state=42, stratify=y)
manifest = pd.read_csv(BASELINE_DIR / "split_manifest.csv", dtype={"sample_id": str})
expected_val = set(manifest.loc[manifest["split"] == "validation", "sample_id"])
assert set(df.iloc[val_idx]["id"]) == expected_val
assert len(train_idx) == 8000 and len(val_idx) == 2000
assert Counter(y[val_idx]) == {0: 1000, 1: 1000}
X_train = texts.iloc[train_idx].reset_index(drop=True)
X_val = texts.iloc[val_idx].reset_index(drop=True)
y_train, y_val = y[train_idx], y[val_idx]
split_id = old_meta["split_id"]
manifest.to_csv(RUN_DIR / "split_manifest.csv", index=False)
cv_splits = list(StratifiedKFold(n_splits=3, shuffle=True, random_state=42).split(X_train, y_train))
fold_manifest = []
for fold, (_, held_idx) in enumerate(cv_splits, 1):
    fold_manifest.extend({"sample_id": df.iloc[train_idx[i]]["id"], "cv_holdout_fold": fold}
                         for i in held_idx)
pd.DataFrame(fold_manifest).to_csv(RUN_DIR / "cv_folds.csv", index=False)
print("Verified original split:", split_id, "| train=8000, validation=2000")

## 2. Baseline error analysis
Fixed bins: <=150, 151–250, >250 whitespace-separated words. Error threshold: P(AI)>=0.5.
Examples are selected by confidence: five false positives and five false negatives when available. Snippets are abbreviated.
These observed examples inform hypotheses; automatic feature statistics do not establish why a prediction failed.

In [ ]:
old_predictions = pd.read_csv(BASELINE_DIR / "validation_predictions.csv",
                              dtype={"sample_id": str}).set_index("sample_id")
val_ids = df.iloc[val_idx]["id"].to_numpy()
baseline_probability = old_predictions.loc[val_ids, "count_word_1_2"].to_numpy()
assert np.array_equal(old_predictions.loc[val_ids, "y_true"].to_numpy(), y_val)
word_lengths = X_val.str.split().str.len().to_numpy()
length_bins = pd.cut(word_lengths, bins=[-np.inf,150,250,np.inf],
                     labels=["<=150", "151-250", ">250"])

def subgroup_metrics(name, probabilities):
    rows = []
    for group in ["<=150", "151-250", ">250"]:
        mask = np.asarray(length_bins == group)
        labels = y_val[mask]
        auc = roc_auc_score(labels, probabilities[mask]) if len(np.unique(labels)) == 2 else np.nan
        rows.append(dict(model=name, length_bin=group, n=int(mask.sum()),
                         n_human=int((labels==0).sum()), n_ai=int((labels==1).sum()),
                         auroc=auc))
    return rows

def error_examples(name, probabilities):
    table = pd.DataFrame({"id":val_ids, "label":y_val, "p_ai":probabilities,
                          "word_count":word_lengths, "text":X_val.to_numpy()})
    fp = table[(table.label==0)&(table.p_ai>=0.5)].sort_values("p_ai",ascending=False).head(5).copy()
    fn = table[(table.label==1)&(table.p_ai<0.5)].sort_values("p_ai").head(5).copy()
    fp["error_type"], fn["error_type"] = "FP", "FN"
    selected = pd.concat([fp,fn],ignore_index=True)
    selected["model"] = name
    selected["snippet"] = selected["text"].str.replace(r"\s+", " ", regex=True).str.slice(0,220)
    selected["failure_hypothesis"] = "Pending human review; snippet alone does not establish cause."
    return selected.drop(columns="text")

baseline_groups = pd.DataFrame(subgroup_metrics("baseline_count",baseline_probability))
baseline_groups.to_csv(RUN_DIR/"baseline_length_metrics.csv",index=False)
baseline_errors = error_examples("baseline_count",baseline_probability)
baseline_errors.to_csv(RUN_DIR/"baseline_error_examples.csv",index=False)
display(baseline_groups)
display(baseline_errors[["id","error_type","p_ai","word_count","snippet"]])
print("Baseline FP:", int(((y_val==0)&(baseline_probability>=0.5)).sum()),
      "| FN:", int(((y_val==1)&(baseline_probability<0.5)).sum()))

## 3. Deterministic style features
No labels or corpus-fitted statistics are used by this transformer.
Features include word/sentence lengths, variability, repetition, punctuation, capitals and digits.
Sentence segmentation is a simple punctuation heuristic (abbreviations can be split incorrectly); these are surface statistics, not universal AI-writing rules.
Style-only is an ablation; counts+style tests incremental value. Count vs binary tests sensitivity to repeated words.

In [ ]:
from rf_features import StyleFeatures

def word_counts(binary=False):
    return CountVectorizer(max_features=20000,ngram_range=(1,2),
                           min_df=2,binary=binary,dtype=np.float32)

representations = {
    "word_count": word_counts(),
    "word_tfidf": TfidfVectorizer(max_features=20000,ngram_range=(1,2),min_df=2,dtype=np.float32),
    "style_only": StyleFeatures(),
    "count_plus_style": FeatureUnion([("counts",word_counts()),("style",StyleFeatures())]),
    "binary_word": word_counts(binary=True),
    "char_tfidf": TfidfVectorizer(analyzer="char",ngram_range=(3,5),max_features=15000,
                                min_df=3,sublinear_tf=True,dtype=np.float32),
}
style_check=StyleFeatures().transform(["", "Hello world! Hello world!"])
assert style_check.shape==(2,len(StyleFeatures.names))
print("Candidates:",list(representations))
print("Style features:",len(StyleFeatures.names))

## 4. Representation selection using training-only cross-validation
Every fold fits its own vectorizer on its training portion. All representations use the same three folds and RF settings.
The selected representation is the one with the highest mean CV AUROC, not highest held-out validation AUROC.
CV standard deviation describes fold variability, not a confidence interval.

In [ ]:
cv_records=[]
for name, prototype in representations.items():
    for fold,(fit_idx,hold_idx) in enumerate(cv_splits,1):
        started=time.perf_counter()
        transformer=clone(prototype)
        A=transformer.fit_transform(X_train.iloc[fit_idx])
        B=transformer.transform(X_train.iloc[hold_idx])
        model=RandomForestClassifier(**RF_BASE)
        model.fit(A,y_train[fit_idx])
        prob=model.predict_proba(B)[:,list(model.classes_).index(1)]
        cv_records.append(dict(representation=name,fold=fold,
                               cv_auroc=roc_auc_score(y_train[hold_idx],prob),
                               seconds=time.perf_counter()-started,n_features=A.shape[1]))
        pd.DataFrame(cv_records).to_csv(RUN_DIR/"representation_cv_folds.csv",index=False)
        print(name,"fold",fold,"AUROC",round(cv_records[-1]["cv_auroc"],6),flush=True)
cv_summary=pd.DataFrame(cv_records).groupby("representation").agg(
    cv_mean=("cv_auroc","mean"),cv_std=("cv_auroc","std"),total_seconds=("seconds","sum")
).sort_values("cv_mean",ascending=False)
cv_summary.to_csv(RUN_DIR/"representation_cv_summary.csv")
selected_representation=cv_summary.index[0]
display(cv_summary)
print("CV-selected representation:",selected_representation)

## 5. Small RF parameter comparison
Only the CV-selected representation is tuned. Four predefined settings are compared on the same training folds.
The default is retained as a candidate, so no improvement is assumed. All use 200 trees and seed 42.
These CV scores have selection bias from searching multiple candidates; the unchanged holdout is checked next.

In [ ]:
settings={
    "base": {},
    "leaf5": {"min_samples_leaf":5},
    "leaf10_depth30": {"min_samples_leaf":10,"max_depth":30},
    "leaf5_features10pct": {"min_samples_leaf":5,"max_features":0.1},
}
tuning_records=[]
for record in cv_records:
    if record["representation"]==selected_representation:
        tuning_records.append(dict(setting="base",fold=record["fold"],cv_auroc=record["cv_auroc"],
                                   seconds=record["seconds"]))
for fold,(fit_idx,hold_idx) in enumerate(cv_splits,1):
    transformer=clone(representations[selected_representation])
    A=transformer.fit_transform(X_train.iloc[fit_idx])
    B=transformer.transform(X_train.iloc[hold_idx])
    for setting,override in settings.items():
        if setting=="base": continue
        started=time.perf_counter()
        model=RandomForestClassifier(**{**RF_BASE,**override})
        model.fit(A,y_train[fit_idx])
        prob=model.predict_proba(B)[:,list(model.classes_).index(1)]
        tuning_records.append(dict(setting=setting,fold=fold,
                                   cv_auroc=roc_auc_score(y_train[hold_idx],prob),
                                   seconds=time.perf_counter()-started))
        pd.DataFrame(tuning_records).to_csv(RUN_DIR/"rf_tuning_folds.csv",index=False)
        print(setting,"fold",fold,"AUROC",round(tuning_records[-1]["cv_auroc"],6),flush=True)
tuning_summary=pd.DataFrame(tuning_records).groupby("setting").agg(
    cv_mean=("cv_auroc","mean"),cv_std=("cv_auroc","std")
).sort_values("cv_mean",ascending=False)
tuning_summary.to_csv(RUN_DIR/"rf_tuning_summary.csv")
selected_setting=tuning_summary.index[0]
selected_params={**RF_BASE,**settings[selected_setting]}
display(tuning_summary)
print("Selected using CV:",selected_representation,selected_setting)

## 6. Holdout evaluation — all candidates, including negative results
Train each representation with baseline RF settings for a controlled comparison. Also evaluate the CV-selected setting.
The holdout ranking is descriptive; it does not replace the predeclared CV selection.
Training time below is model.fit only; feature and prediction times are recorded separately.

In [ ]:
experiment_results=[]
val_predictions=pd.DataFrame({"sample_id":val_ids,"y_true":y_val})
fitted_models={}
fitted_transformers={}

def evaluate_candidate(name,prototype,params):
    started=time.perf_counter()
    transformer=clone(prototype)
    A=transformer.fit_transform(X_train)
    B=transformer.transform(X_val)
    feature_seconds=time.perf_counter()-started
    model=RandomForestClassifier(**params)
    started=time.perf_counter()
    model.fit(A,y_train)
    fit_seconds=time.perf_counter()-started
    started=time.perf_counter()
    positive=list(model.classes_).index(1)
    train_prob=model.predict_proba(A)[:,positive]
    val_prob=model.predict_proba(B)[:,positive]
    predict_seconds=time.perf_counter()-started
    train_auc=roc_auc_score(y_train,train_prob)
    val_auc=roc_auc_score(y_val,val_prob)
    experiment_results.append(dict(
        experiment=name,train_auroc=train_auc,val_auroc=val_auc,
        gap=train_auc-val_auc,n_features=A.shape[1],
        feature_seconds=feature_seconds,fit_seconds=fit_seconds,predict_seconds=predict_seconds,
        params=json.dumps(params,sort_keys=True),split_id=split_id,
    ))
    val_predictions[name]=val_prob
    fitted_models[name]=model
    fitted_transformers[name]=transformer
    pd.DataFrame(experiment_results).to_csv(RUN_DIR/"holdout_results.csv",index=False)
    val_predictions.to_csv(RUN_DIR/"validation_predictions.csv",index=False)
    print(name,": train",round(train_auc,6),"validation",round(val_auc,6),flush=True)

for name,prototype in representations.items():
    evaluate_candidate(name,prototype,RF_BASE)
selected_name=selected_representation
if selected_setting!="base":
    selected_name=selected_representation+"__"+selected_setting
    evaluate_candidate(selected_name,representations[selected_representation],selected_params)
assert np.allclose(val_predictions["word_count"],baseline_probability,rtol=0,atol=1e-12), "Baseline reproduction changed."
result_table=pd.DataFrame(experiment_results).sort_values("val_auroc",ascending=False)
display(result_table[["experiment","train_auroc","val_auroc","gap","n_features","fit_seconds"]])
print("CV-selected model (not selected from holdout ranking):",selected_name)

## 7. Follow-up diagnostics and uncertainty
Recompute length-bin metrics and confidence-selected errors for the selected model.
Paired stratified bootstrap resamples the same validation examples for baseline and selected predictions, keeping 1,000 examples of each class in each replicate.
The interval describes sampling uncertainty conditional on these models/data; it does not prove robustness to unseen domains or generators.

In [ ]:
selected_probability=val_predictions[selected_name].to_numpy()
group_rows=subgroup_metrics("baseline_count",baseline_probability)+subgroup_metrics(selected_name,selected_probability)
pd.DataFrame(group_rows).to_csv(RUN_DIR/"length_metrics_comparison.csv",index=False)
errors=error_examples(selected_name,selected_probability)
errors.to_csv(RUN_DIR/"selected_error_examples.csv",index=False)
display(pd.DataFrame(group_rows))
display(errors[["id","error_type","p_ai","word_count","snippet"]])
rng=np.random.default_rng(42)
negative_idx=np.flatnonzero(y_val==0); positive_idx=np.flatnonzero(y_val==1)
deltas=[]
for _ in range(1000):
    idx=np.concatenate([rng.choice(negative_idx,len(negative_idx),replace=True),
                        rng.choice(positive_idx,len(positive_idx),replace=True)])
    deltas.append(roc_auc_score(y_val[idx],selected_probability[idx])-
                  roc_auc_score(y_val[idx],baseline_probability[idx]))
delta=float(roc_auc_score(y_val,selected_probability)-roc_auc_score(y_val,baseline_probability))
ci=np.quantile(deltas,[0.025,0.975]).tolist()
uncertainty={"selected_model":selected_name,"delta_vs_count_baseline":delta,
             "paired_bootstrap_95pct_interval":ci,"resamples":1000,"seed":42}
(RUN_DIR/"bootstrap_comparison.json").write_text(json.dumps(uncertainty,indent=2),encoding="utf-8")
print(uncertainty)

## 8. Save a report-ready factual experiment log
The record distinguishes observations from hypotheses and retains unsuccessful attempts.
No causal explanations are automatically assigned to error examples.
This notebook addresses model comparison and a controlled improvement; it does not complete the entire course report.

In [ ]:
versions={"python":sys.version,"numpy":np.__version__,"pandas":pd.__version__,
          "scipy":scipy.__version__,"scikit-learn":sklearn.__version__}
metadata={"dataset_sha256":dataset_hash,"split_id":split_id,
          "validation_size":2000,"split_seed":42,"cv_seed":42,"cv_folds":3,
          "selected_representation":selected_representation,"selected_setting":selected_setting,
          "selected_model":selected_name,"selected_params":selected_params,"versions":versions,
          "candidate_feature_params":{name:repr(obj) for name,obj in representations.items()},
          "selection":"training-only CV; validation already used for exploratory diagnostics",
          "test_usage":"No test data used in model selection; no Kaggle submission.",
          "style_tokenization":"English alphabetic words with optional apostrophe; simple punctuation sentence splitting"}
(RUN_DIR/"run_metadata.json").write_text(json.dumps(metadata,indent=2,default=str),encoding="utf-8")
(PROJECT_DIR/"requirements_rf.txt").write_text(
    "\n".join(f"{k}=={versions[k]}" for k in ["numpy","pandas","scipy","scikit-learn"])+"\n",
    encoding="utf-8")
lines=[
    "# Random Forest improvement results","",
    f"Run: {RUN_DIR.name}",f"Split: {split_id} (8,000 training / 2,000 validation).","",
    "## Protocol",
    "Representation selection used 3-fold stratified CV within the training partition. Vocabulary and IDF were refitted inside each fold. Four RF settings were compared for the CV-selected representation. Validation IDs remained fixed.",
    "Baseline validation errors informed the hypotheses; therefore holdout results are exploratory.",
    "",
    "## Representation comparison",
    "| Experiment | Train AUROC | Validation AUROC | Gap | Fit seconds |",
    "| --- | ---: | ---: | ---: | ---: |",
]
for row in experiment_results:
    lines.append(f"| {row['experiment']} | {row['train_auroc']:.6f} | {row['val_auroc']:.6f} | {row['gap']:.6f} | {row['fit_seconds']:.3f} |")
lines+=["","## Selection and uncertainty",
        f"Training CV selected **{selected_name}**.",
        f"Validation AUROC change versus original word-count RF: {delta:+.6f}.",
        f"Paired bootstrap 95% percentile interval: [{ci[0]:+.6f}, {ci[1]:+.6f}] (1,000 resamples).",
        "This interval does not account for all development/selection decisions and does not establish Kaggle generalization.",
        "","## Interpretation boundaries",
        "- Preserve all candidate results, including regressions. Do not present only the best validation score.",
        "- Style features and character features are hypotheses about complementary information; improved scores do not establish a causal mechanism.",
        "- Length-bin AUROC and error IDs are saved separately. Failure hypotheses require reading the examples in context.",
        "- No test-label access, external labels, or Kaggle submission was used.",
        "- The teammate's local/Kaggle gap remains unexplained without their code, split, and predictions.",
        "- Frozen embeddings, unlabeled-data experiments, learning curves, and teammate ensembling remain future work.",
        ""]
(RUN_DIR/"EXPERIMENT_REPORT.md").write_text("\n".join(lines),encoding="utf-8")
print("Saved report and all results:",RUN_DIR)